# 📊 Week 1 — Exploratory Data Analysis (EDA): Reading Guide

**Learning Objectives (Week 1 – EDA)**  
- Understand the motivation for MLOps and how EDA fits into a production ML lifecycle.  
- Connect to Redshift and perform reproducible EDA.  
- Document data quality issues and define target/feature schema.  
- Prepare train/validation/test splits with leakage-aware methodology.  

> **Context**: ZAP is targeting **MLOps Level 2**. Even EDA should be reproducible and versioned (data query, sampling, and preprocessing code committed).

## 🔍 What is EDA and Why It Matters
Exploratory Data Analysis (EDA) is the process of **exploring, visualizing, and validating datasets** before training models.  
In **MLOps**, EDA is about much more than plots — it’s about **data reliability** and ensuring downstream pipelines are stable.

**Why it matters for production:**
- 🗑️ **Garbage in, garbage out** → poor data = poor models.  
- ⚡ **Operational resilience** → detect defects early, before they hit production.  
- 🔁 **Pipeline reliability** → schemas and checks from EDA become the foundation for automation.  


## 📐 Data Quality Dimensions
Checking data quality ensures your model won’t collapse when facing real-world inputs. Here are the key dimensions:

| Dimension    | Question to Ask | Example Issue |
|--------------|-----------------|---------------|
| ✅ Completeness | Are required values present? | Missing customer age |
| 🔄 Consistency | Do values follow expected formats/relations? | Country code "PT" inconsistently mapped |
| 🎯 Accuracy | Are values correct? | Negative product price |
| 🧩 Validity | Do values conform to rules/types? | Dates stored as free-text |
| ⏱️ Timeliness | Is the data up to date? | Using last year’s sales for today’s forecast |


## ⚠️ Leakage and Target Contamination
- **Data leakage** → using information not available at prediction time.  
- **Target contamination** → when the target leaks into features or data splits.  

❌ Example leakage: Using "credit approval status" as a feature to predict loan approval.  
❌ Example contamination: Randomly splitting time-series data, letting future events “leak” into training.

➡️ Both lead to inflated metrics **during training** and catastrophic failures **in production**.


## ♻️ Reproducibility
Reproducibility = **same results given same inputs**. Essential for trust, debugging, and collaboration.

Key practices:
- 🎲 **Fixed seeds** → ensure reproducible sampling/splitting.  
- 📑 **Deterministic queries** → e.g., always `ORDER BY id` in SQL.  
- 🖥️ **Environment capture** → record Python & library versions, OS, hardware.  

Without reproducibility → experiments can’t be compared, bugs can’t be traced.


## 📦 Outputs That Feed the Pipeline
EDA is not a one-off. Its **outputs become artifacts** for the ML pipeline:

- 🗂️ **Feature schema** → defines types, ranges, categories, nullability.  
- ✅ **Data checks** → rules like “no nulls in IDs” or “target is binary.”  
- ✂️ **Split strategy** → deterministic, leakage-free train/val/test partitions.  

These artifacts support:
- Automation in CI/CD ✅  
- Monitoring in production 📈  
- MLOps Level 2 maturity ⚙️  

---

# 📝 Exercises - Build the Dataset (Cont.)

You should choose any dataset existing on Redshift to practice EDA, and gather relevant information to train your model.  
Dataset example should contain customer demographics, services, account info, etc.

## Import needed libraries

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

w = 800

In [ ]:
%pip install awswrangler

## 🔧 Setup

### Data Loading Configuration

Use the function `load_data()` provided in `data_io.py` to load data from:
- **Local/S3 parquet** (for development/testing)
- **Redshift via UNLOAD** (using boto3 redshift-data API - matches test_redshift_connection.ipynb)

### 📝 Instructions

**⚠️ IMPORTANT: Before running the next cell:**

1. **Change `USER_NAME`** to your name (e.g., `"john_doe"`, `"maria_silva"`)
   - This creates a separate S3 folder for each student
   - Prevents overwriting other students' data
   - Example path: `s3://bucket/redshift_exports/john_doe/eda_train_2025-10-22/`

2. **Run the cell** to load data from Redshift

The configuration uses the **exact same pattern** as `test_redshift_connection.ipynb`:
- boto3 redshift-data API (simpler, no direct connection needed)
- Same credentials from test notebook
- Same UNLOAD pattern to S3 as Parquet
- awswrangler for reading Parquet files

**Alternative:** Switch to local parquet for quick testing by commenting out OPTION 2 and uncommenting OPTION 1.

⚠️ **Note**: Never commit credentials to git! Use `.gitignore` or environment variables in production.

In [ ]:
# ========================================
# Data Loading Options
# ========================================
from data_io import load_data
from datetime import datetime, UTC, date

# OPTION 1: Load from local parquet (for development/testing)
# ========================================
# Uncomment to load from local parquet file:
#
# df = load_data(
#     source='parquet',
#     uri='../sample_data_from_redshift/sample.parquet'
# )

# OPTION 2: Load from Redshift via UNLOAD (boto3 redshift-data API - DEFAULT)
# ========================================
# This follows the EXACT same pattern as test_redshift_connection.ipynb:
# 1. Use boto3 redshift-data API (no direct connection needed)
# 2. Execute UNLOAD to export data to S3 as Parquet
# 3. Poll for query completion
# 4. Read Parquet files from S3 using awswrangler

# ⚠️ TODO: CHANGE THIS TO YOUR NAME (to avoid overwriting other students' data)
# ================================================
USER_NAME = "federico"  # ⚠️ CHANGE THIS! Example: "john_doe"

# Redshift and S3 Configuration (matches test_redshift_connection.ipynb)
# ================================================
REDSHIFT_CONFIG = {
    'cluster_id': 'redshift-cluster-dsi',
    'database': 'prod',
    'db_user': 'svc_sagemaker',
    'region': 'af-south-1',
    'iam_role': 'arn:aws:iam::733246370304:role/RedshiftIAMAuthRole',
    's3_export_prefix': f"s3://sagemaker-af-south-1-733246370304/redshift_exports/{USER_NAME}/eda_train_{date.today():%Y-%m-%d}",
    'cleanup_s3': False  # Set to True to delete temp files after loading
}

# SQL Query - Use training features with RANDOM() sampling for EDA
# ================================================
SQL_QUERY = """
SELECT *
FROM dth_churn_ml_training.training_features
WHERE RANDOM() < 0.0001;
"""

# Load data using boto3 redshift-data API pattern (same as test notebook)
# Sampling is handled by RANDOM() in the SQL query
df = load_data(
    source='redshift',
    sql=SQL_QUERY,
    redshift_kwargs=REDSHIFT_CONFIG
)

# Convert date columns
date_cols = ['iddim_date_inicio', 'iddim_date_fim']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"S3 Export Path: {REDSHIFT_CONFIG['s3_export_prefix']}")

# Data provenance - track metadata for MLOps
metadata = {
    "source": "redshift",
    "user": USER_NAME,
    "extraction_date": datetime.now(UTC).isoformat(),
    "n_rows": len(df),
    "n_columns": len(df.columns),
    "date_range": {
        "min": str(df['iddim_date_inicio'].min()) if 'iddim_date_inicio' in df.columns else None,
        "max": str(df['iddim_date_inicio'].max()) if 'iddim_date_inicio' in df.columns else None
    },
    "version": "v1.0"
}

print(f"\n📋 Data Provenance:")
print(f"  User: {metadata['user']}")
print(f"  Source: {metadata['source']}")
print(f"  Extracted: {metadata['extraction_date'][:10]}")
print(f"  Date range: {metadata['date_range']['min']} to {metadata['date_range']['max']}")

## 5. Bivariate Analysis

**Goal**: Understand how features relate to the target (churn)

**Strategy**:
1. Calculate feature-target correlations (already done ✓)
2. Auto-plot **top 5 numeric** features vs churn (box plots)
3. Auto-plot **top 5 categorical** features vs churn (stacked bars)
4. Manual exploration available for specific comparisons

In [ ]:
# Feature correlation with target
from scipy.stats import pointbiserialr, chi2_contingency

def cramers_v(x, y):
    """Cramér's V for categorical features."""
    crosstab = pd.crosstab(x, y)
    chi2 = chi2_contingency(crosstab)[0]
    n = crosstab.sum().sum()
    return np.sqrt(chi2 / (n * (min(crosstab.shape) - 1)))

def feature_target_correlation(df, target='churn'):
    """Calculate correlation between features and target (excludes IDs and datetime)."""
    results = []
    
    # Exclude ID columns, datetime columns, and target from analysis
    id_cols = [col for col in df.columns if 'id' in col.lower() or 'codigo' in col.lower()]
    datetime_cols = df.select_dtypes(include='datetime').columns.tolist()
    exclude_cols = id_cols + datetime_cols + [target]
    
    for col in df.columns:
        if col in exclude_cols or df[col].nunique() <= 1:
            continue
        
        if pd.api.types.is_numeric_dtype(df[col]):
            # Numeric: point-biserial correlation
            corr, _ = pointbiserialr(df[col].dropna(), df.loc[df[col].notna(), target])
            results.append({'feature': col, 'correlation': abs(corr), 'method': 'point-biserial'})
        else:
            # Categorical: Cramér's V
            corr = cramers_v(df[col].astype(str), df[target])
            results.append({'feature': col, 'correlation': corr, 'method': 'cramers_v'})
    
    return pd.DataFrame(results).sort_values('correlation', ascending=False).reset_index(drop=True)

print("✓ Feature-target correlation function defined (excludes ID columns and datetime)")

### 🎯 Feature-Target Correlation

Calculate how strongly each feature correlates with churn. This will guide our bivariate visualizations.

In [ ]:
# Bivariate visualization helpers

def plot_numeric_vs_churn(df, col):
    """Box plot: numeric feature by churn."""
    fig = px.box(
        df, x='churn', y=col, color='churn',
        title=f'{col} by Churn',
        width=w, height=w/1.5
    )
    fig.show()

def plot_categorical_vs_churn(df, col, top_k=10):
    """Stacked bar: churn rate within each category."""
    # Keep top K categories
    top_cats = df[col].value_counts().head(top_k).index
    plot_df = df[df[col].isin(top_cats)].copy()
    
    # Calculate churn rate per category
    counts = plot_df.groupby([col, 'churn']).size().reset_index(name='count')
    counts['pct'] = counts.groupby(col)['count'].transform(lambda x: x / x.sum())
    
    fig = px.bar(
        counts, x=col, y='pct', color='churn',
        title=f'Churn Rate by {col} (top {top_k})',
        barmode='stack',
        labels={'pct': 'Proportion'},
        width=w, height=w/1.5
    )
    fig.update_yaxes(tickformat=".0%")
    fig.show()

print("✓ Bivariate helper functions defined")

In [ ]:
# Calculate feature-target correlations FIRST
corr_df = feature_target_correlation(df, target='churn')

print("Top 15 features most correlated with churn:\n")
print(corr_df.head(15).to_string(index=False))

print(f"\n💡 Interpretation:")
print("  • Cramér's V: categorical features (0 = no association, 1 = perfect)")
print("  • Point-biserial: numeric features (-1 to +1, like Pearson)")
print(f"\n  Most correlations are <0.15 (weak to moderate) - typical for churn datasets")

# Auto-plot TOP 5 NUMERIC features
top_numeric = corr_df[corr_df['method'] == 'point-biserial'].head(5)

print(f"\n📊 Top {len(top_numeric)} Numeric Features vs Churn:\n")
for idx, row in top_numeric.iterrows():
    print(f"  {row['feature']} (correlation: {row['correlation']:.3f})")
    plot_numeric_vs_churn(df, row['feature'])

In [ ]:
# Auto-plot TOP 5 CATEGORICAL features
top_categorical = corr_df[corr_df['method'] == 'cramers_v'].head(5)

print(f"\n📊 Top {len(top_categorical)} Categorical Features vs Churn:\n")
for idx, row in top_categorical.iterrows():
    print(f"  {row['feature']} (Cramér's V: {row['correlation']:.3f})")
    plot_categorical_vs_churn(df, row['feature'], top_k=10)

### 🔍 Manual Bivariate Exploration

Use the helper functions to explore any feature-target relationship:

In [ ]:
# Uncomment to explore specific feature-target relationships:
# plot_numeric_vs_churn(df, 'tenure_days')
# plot_numeric_vs_churn(df, 'age')
# plot_categorical_vs_churn(df, 'provincia', top_k=15)
# plot_categorical_vs_churn(df, 'sexo')

## 7. Reproducibility Practices
- Set a **random seed** when sampling rows for inspection.  
- Export a **feature schema JSON** with column names, types, and allowed ranges/categories.
- Save **metadata** about data provenance for MLOps tracking.

In [ ]:
# Reproducible sampling with fixed seed
df_sample = df.sample(500, random_state=42)
print(f"Sampled {len(df_sample)} rows (always the same rows with random_state=42)")

In [ ]:
# Export feature schema
import json

schema = {}

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        schema[col] = {
            "type": "numeric",
            "min": float(df[col].min()),
            "max": float(df[col].max())
        }
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        schema[col] = {
            "type": "datetime",
            "min": str(df[col].min()),
            "max": str(df[col].max())
        }
    else:
        schema[col] = {
            "type": "categorical",
            "categories": df[col].dropna().unique().tolist()[:50]
        }

with open("feature_schema.json", "w") as f:
    json.dump(schema, f, indent=2)

print("✓ Feature schema saved to feature_schema.json")

In [ ]:
# Save complete metadata
metadata['feature_count'] = {
    'numeric': df.select_dtypes(include='number').shape[1],
    'categorical': df.select_dtypes(include='object').shape[1],
    'datetime': df.select_dtypes(include='datetime').shape[1]
}

metadata['target_distribution'] = {
    'class_0': int(df['churn'].value_counts()[0]),
    'class_1': int(df['churn'].value_counts()[1]),
    'imbalance_ratio': float(df['churn'].value_counts()[0] / df['churn'].value_counts()[1])
}

metadata['data_quality'] = {
    'missing_values': int(df.isna().sum().sum()),
    'duplicate_rows': int(df.duplicated().sum())
}

with open("data_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✓ Metadata saved to data_metadata.json")

## 8. Train/Validation/Test Split Strategy
- Propose and implement a split strategy:
  - **Time-based split** to prevent data leakage (train on past, validate/test on future).
  - Sort by date before splitting to ensure temporal ordering.
  - Document why time-based splitting is necessary here (no future information leaking into training).

In [ ]:
# Time-based split (prevents data leakage)
# Sort by date to ensure train comes before validation/test
df_sorted = df.sort_values('iddim_date_inicio')

# Split: 80% train, 10% val, 10% test
n = len(df_sorted)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

train_df = df_sorted.iloc[:train_end]
val_df = df_sorted.iloc[train_end:val_end]
test_df = df_sorted.iloc[val_end:]

print("Split sizes:")
print(f"  Train: {len(train_df):,} ({len(train_df)/n:.1%})")
print(f"  Val:   {len(val_df):,} ({len(val_df)/n:.1%})")
print(f"  Test:  {len(test_df):,} ({len(test_df)/n:.1%})")

print("\nDate ranges:")
print(f"  Train: {train_df['iddim_date_inicio'].min()} to {train_df['iddim_date_inicio'].max()}")
print(f"  Val:   {val_df['iddim_date_inicio'].min()} to {val_df['iddim_date_inicio'].max()}")
print(f"  Test:  {test_df['iddim_date_inicio'].min()} to {test_df['iddim_date_inicio'].max()}")

# 🎯 Deliverables
By the end of these exercises, you should have:
1. A **data dictionary**.  
2. Summary tables/plots of findings and key features.  
3. A **feature schema JSON** with data types and constraints.  
4. A **train/val/test split file** (e.g., `splits.json`) for reproducible downstream tasks.  

## Peer Validation
  - Reproducible data loading (query or seed).  
  - Clear schema with rationale per feature.  
  - Split method documented and leakage-safe.  
  - Artifacts present and versioned.

## 9. Optional: Automated EDA Report

For a comprehensive automated analysis, you can use ydata-profiling (formerly pandas-profiling).

In [ ]:
%pip install ydata-profiling

In [ ]:
from ydata_profiling import ProfileReport
profile = ProfileReport(df, title="EDA Report — Customer Churn", explorative=True)
profile.to_file("eda_report.html")

print("💡 Tip: Install ydata-profiling for automated EDA reports")